In [ ]:
import requests
from aind_data_transfer_service.models.core import (
    UploadJobConfigsV2,
    SubmitJobRequestV2,
    Task,
)
from aind_data_schema_models.modalities import Modality
from datetime import datetime
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "project_config.ini").is_file():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find project_config.ini from the notebook working directory.")
    PROJECT_ROOT = parent
sys.path.insert(0, str(PROJECT_ROOT / "code"))

## Upload specific sessions

In [ ]:
from glob import glob

import requests
from aind_data_transfer_service.models.core import (
    UploadJobConfigsV2,
    SubmitJobRequestV2,
    Task,
)

from aind_data_schema_models.modalities import Modality

from datetime import datetime
import json
import os

def upload_single_session_ephys(folder_path, file_name):
    patterns = [
        'acquisition_ecephys.json',
    ]

    root_folder = (
        os.path.join(folder_path, file_name)
    )
    
    session_data = None

    for pattern in patterns:
        matches = sorted(glob(os.path.join(root_folder, pattern)))
        if matches:
            with open(matches[0], 'r') as f:
                session_data = json.load(f)
            break

    if session_data is None:
        print(
            f"No session or acquisition file found in {root_folder}. "
            "Expected acquisition_ecephys.json."
        )
        return
    
    # The vr_foraging job_type contains presets for compression and Code Ocean pipelines.
    # Please see http://aind-data-transfer-service/job_params for list of job_types.
    job_type = "vr_foraging_v2"
    subject_id = session_data['subject_id']

    try:
        acq_datetime = datetime.fromisoformat(session_data['session_start_time'])
    except KeyError:
        acq_datetime = datetime.fromisoformat(session_data['acquisition_start_time'])

    project_name = "Cognitive flexibility in patch foraging"
    s3_bucket = "default"


    metadata_dir = root_folder

    gather_preliminary_metadata = Task(
        job_settings={
            "metadata_dir": metadata_dir,
        }
    )

    behavior_videos_source = (
        f"{root_folder}/behavior-videos"
    )
    behavior_source = (
        f"{root_folder}/behavior"
    )

    ephys_source = (
        f"{root_folder}/ecephys"
    )
    
    behavior_videos_task = Task(
        job_settings={
            "input_source": behavior_videos_source
        }
    )
    behavior_task = Task(
        job_settings={
            "input_source": behavior_source
        }
    )

    ephys_task = Task(
        job_settings={
            "input_source": ephys_source,
            "check_timestamps": False # Optional setting to check for timestamp consistency in ecephys data
        }
    )

    modalities = [
                  Modality.BEHAVIOR, 
                  Modality.ECEPHYS,
                  Modality.BEHAVIOR_VIDEOS
                  ]

    modality_transformation_settings = {
        "behavior": behavior_task,
        "behavior-videos": behavior_videos_task,
        "ecephys": ephys_task,
    }
        
    upload_job_configs_v2 = UploadJobConfigsV2(
        job_type=job_type,
        user_email="tiffany.ona@alleninstitute.org",
        project_name=project_name,
        s3_bucket=s3_bucket,
        subject_id=subject_id,
        acq_datetime=acq_datetime,
        modalities=modalities,
        tasks={
            "gather_preliminary_metadata": gather_preliminary_metadata,
            "modality_transformation_settings": modality_transformation_settings,
            "remove_source_folders": {"skip_task": True}, # does not remove source data
            
        },
    )

    # Add more to the list if needed
    upload_jobs = [upload_job_configs_v2]

    # Optional email address and notification types if desired
    submit_request_v2 = SubmitJobRequestV2(
        user_email="tiffany.ona@alleninstitute.org",  # <-- add here
        upload_jobs=upload_jobs
    )
    post_request_content = submit_request_v2.model_dump(mode="json", exclude_none=True)

    # Please use the production endpoint for submitting jobs and the dev endpoint
    # for running tests.
    endpoint = "http://aind-data-transfer-service"
    # endpoint = "http://aind-data-transfer-service-dev"  # For testing

    # Optionally validate the job request before submitting
    validate_job_response = requests.post(
        url=f"{endpoint}/api/v2/validate_json",
        json=post_request_content,
    )
    # print(validate_job_response.status_code)
    print(validate_job_response.json())

    # Submit the job request
    submit_job_response = requests.post(
        url=f"{endpoint}/api/v2/submit_jobs",
        json=post_request_content,
    )

In [ ]:
from project_config import SESSIONS_DIR
folder_path = str(SESSIONS_DIR)
file_name = '798279_2025-12-01_12-11-35'

upload_single_session_ephys(folder_path, file_name)

{'message': 'Valid model', 'data': {'version': '2.2.2', 'model_json': {'dag_id': 'transform_and_upload_v2', 'user_email': 'tiffany.ona@alleninstitute.org', 'email_notification_types': ['fail'], 'upload_jobs': [{'job_type': 'vr_foraging_v2', 'user_email': 'tiffany.ona@alleninstitute.org', 'email_notification_types': ['fail'], 's3_bucket': 'default', 'project_name': 'Cognitive flexibility in patch foraging', 'modalities': [{'name': 'Behavior', 'abbreviation': 'behavior'}, {'name': 'Extracellular electrophysiology', 'abbreviation': 'ecephys'}], 'subject_id': '798279', 'acq_datetime': '2025-12-01T12:11:35Z', 'tasks': {'gather_preliminary_metadata': {'skip_task': False, 'job_settings': {'metadata_dir': '//allen/aind/scratch/tiffany.ona/ephys/sessions/798279_2025-12-01_12-11-35'}}, 'modality_transformation_settings': {'behavior': {'skip_task': False, 'job_settings': {'input_source': '//allen/aind/scratch/tiffany.ona/ephys/sessions/798279_2025-12-01_12-11-35/behavior'}}, 'ecephys': {'skip_tas